In [15]:
# Auto-reload python code on change
%load_ext autoreload
%autoreload 2

from word2vec import *
import numpy as np

import matplotlib.pyplot as plt


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
text = "The cat sat on the mat."

tokens = tokenize(text)

print(tokens)


['the', 'cat', 'sat', 'on', 'the', 'mat']


In [5]:
corpus = "The cat sat on the mat. The dog sat on the rug. The cat chased the mouse. The dog chased the cat."

sentences = split_sentences(corpus)

tokenized_sentences = []

for sentence in sentences:
    tokens = tokenize(sentence)
    tokenized_sentences.append(tokens)

print(tokenized_sentences)


[['the', 'cat', 'sat', 'on', 'the', 'mat'], ['the', 'dog', 'sat', 'on', 'the', 'rug'], ['the', 'cat', 'chased', 'the', 'mouse'], ['the', 'dog', 'chased', 'the', 'cat']]


In [6]:
id_to_word, word_to_id, word_counts = build_vocab(tokenized_sentences)

print(id_to_word)
print(word_to_id)
print(word_counts)


{0: 'cat', 1: 'chased', 2: 'dog', 3: 'mat', 4: 'mouse', 5: 'on', 6: 'rug', 7: 'sat', 8: 'the'}
{'cat': 0, 'chased': 1, 'dog': 2, 'mat': 3, 'mouse': 4, 'on': 5, 'rug': 6, 'sat': 7, 'the': 8}
[3, 2, 2, 1, 1, 2, 1, 2, 8]


In [7]:
pairs = generate_pairs(tokenized_sentences, word_to_id)

print(pairs)


[(8, 0), (8, 7), (0, 8), (0, 7), (0, 5), (7, 8), (7, 0), (7, 5), (7, 8), (5, 0), (5, 7), (5, 8), (5, 3), (8, 7), (8, 5), (8, 3), (3, 5), (3, 8), (8, 2), (8, 7), (2, 8), (2, 7), (2, 5), (7, 8), (7, 2), (7, 5), (7, 8), (5, 2), (5, 7), (5, 8), (5, 6), (8, 7), (8, 5), (8, 6), (6, 5), (6, 8), (8, 0), (8, 1), (0, 8), (0, 1), (0, 8), (1, 8), (1, 0), (1, 8), (1, 4), (8, 0), (8, 1), (8, 4), (4, 1), (4, 8), (8, 2), (8, 1), (2, 8), (2, 1), (2, 8), (1, 8), (1, 2), (1, 8), (1, 0), (8, 2), (8, 1), (8, 0), (0, 1), (0, 8)]


In [8]:
vocab_size = len(id_to_word)
input_embeddings, output_embeddings = initialize_embeddings(vocab_size, 3)

print("input shape:", input_embeddings.shape)
print("output shape:", output_embeddings.shape)
print(input_embeddings)


input shape: (9, 3)
output shape: (9, 3)
[[ 0.03047171 -0.10399841  0.07504512]
 [ 0.09405647 -0.19510352 -0.13021795]
 [ 0.01278404 -0.03162426 -0.00168012]
 [-0.08530439  0.0879398   0.07777919]
 [ 0.00660307  0.11272412  0.04675093]
 [-0.08592925  0.03687508 -0.09588826]
 [ 0.08784503 -0.00499259 -0.01848624]
 [-0.06809295  0.12225413 -0.01545295]
 [-0.04283278 -0.03521336  0.05323092]]


In [9]:
cat_id = word_to_id["cat"]
sat_id = word_to_id["sat"]

cat_vector = input_embeddings[cat_id]
sat_output_vector = output_embeddings[sat_id]

score = cat_vector @ sat_output_vector

print("Cat input vector:", cat_vector)
print("Sat output vector:", sat_output_vector)
print("Score:", score)


Cat input vector: [ 0.03047171 -0.10399841  0.07504512]
Sat output vector: [0.06789136 0.00675791 0.02891194]
Score: 0.0035356539863971716


In [10]:
scores = calculate_scores(word_to_id['cat'], input_embeddings, output_embeddings)

print(scores.shape)


(9,)


In [11]:
input_embeddings, output_embeddings = initialize_embeddings(vocab_size, 3)

scores_before = calculate_scores(cat_id, input_embeddings, output_embeddings)
probabilities_before = softmax(scores_before)
loss_before = cross_entropy(scores_before, sat_id)

print("Target probability before:", probabilities_before[sat_id])
print("Loss before:", loss_before)



Target probability before: 0.11131687927250628
Loss before: 2.1953743765612197


In [12]:
center_vector = input_embeddings[cat_id]

center_gradient, output_gradients = backward(
    probabilities_before,
    sat_id,
    center_vector,
    output_embeddings,
)

print("Center gradient:", center_gradient)
print("Center gradient shape:", center_gradient.shape)
print("Output gradients shape:", output_gradients.shape)


Center gradient: [-0.03328468 -0.00448069 -0.01711386]
Center gradient shape: (3,)
Output gradients shape: (9, 3)


In [13]:
learning_rate = 0.1

input_embeddings[cat_id] -= learning_rate * center_gradient
output_embeddings -= learning_rate * output_gradients

scores_after = calculate_scores(cat_id, input_embeddings, output_embeddings)
probabilities_after = softmax(scores_after)
loss_after = cross_entropy(scores_after, sat_id)

print(
    "Target probability:",
    probabilities_before[sat_id],
    "->",
    probabilities_after[sat_id],
)
print("Loss:", loss_before, "->", loss_after)

assert probabilities_after[sat_id] > probabilities_before[sat_id]
assert loss_after < loss_before


Target probability: 0.11131687927250628 -> 0.11150649921128168
Loss: 2.1953743765612197 -> 2.1936724008902764


In [16]:
input_embeddings, output_embeddings, loss_history = train(
    pairs=pairs,
    vocab_size=len(word_to_id),
    embedding_dim=5,
    epochs=200,
    learning_rate=0.01,
)

print(f"first epoch loss: {loss_history[0]}")
print(f"second epoch loss: {loss_history[-1]}")



first epoch loss: 2.1984483322681
second epoch loss: 1.5447359796512812
